data.rename(columns={
    "What best describes the place you grew up in? ": "Place_Grew_Up",
    "When considering a purchase, how important are the following factors to you?  [Price/Cost]": "Price_Importance",
    "When considering a purchase, how important are the following factors to you?  [Brand reputation]": "Brand_Importance",
    "When considering a purchase, how important are the following factors to you?  [Peer recommendation]": "Peer_Importance",
    "When considering a purchase, how important are the following factors to you?  [Long-term utility/value]": "Utility_Importance"
}, inplace=True)


data.rename(columns={
    "How do you track your monthly expenditures? ": "Track_Expenditures",
    "What is the expected graph for your expenditure for the months ": "Expenditure_Graph"
}, inplace=True)


data.rename(columns={
    "In which of the following scenarios would you justify an unexpected expense of ₹1,500 or more?": "Justify_Unexpected_Expense"
}, inplace=True)

data.rename(columns={
    "How often do you make purchases that you hadn’t planned for? ": "Unplanned_Purchases",
    "On average, how much do you spend per month (excluding tuition fees)? ": "Monthly_Spend",
    "On a scale of 1 to 5, how much do social events or peer pressure influence your spending? ": "Peer_Influence",
    "How confident do you feel in your ability to manage your personal finances? ": "Finance_Confidence"
}, inplace=True)

data.rename(columns={
    "In which categories do you spend the majority of your budget?  [Food & Dining]": "Budget_FoodDining",
    "In which categories do you spend the majority of your budget?  [Travel]": "Budget_Travel",
    "In which categories do you spend the majority of your budget?  [Fashion]": "Budget_Fashion",
    "In which categories do you spend the majority of your budget?  [Subscriptions (Netflix, Spotify, etc.)]": "Budget_Subscriptions",
    "In which categories do you spend the majority of your budget?  [Fun & Entertainment]": "Budget_Entertainment"
}, inplace=True)


groups = []

for i in data["Monthly_Spend"]:
    if i == 3 or i == 1 or i == 2:
        groups.append("Low Spenders")
    elif i == 6 or i == 4 or i == 5:
        groups.append("Medium Spenders")
    else:
        groups.append("High Spenders")
data["Group"] = groups

budget_cols = ["Budget_FoodDining", "Budget_Travel","Budget_Fashion", "Budget_Subscriptions","Budget_Entertainment"]
data[budget_cols] = data[budget_cols].replace("Answer", "Yes").fillna("No")

data["Expenditure_Graph"] = data["Expenditure_Graph"].astype(str)
data.drop(columns=["Justify_Unexpected_Expense"], inplace=True)

In [8]:
import pandas as pd
from kmodes.kprototypes import KPrototypes

In [9]:
data = pd.read_csv("../Dataset/Scored_Categorical_Data.csv")

In [10]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 507 entries, 0 to 506
Data columns (total 20 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Unnamed: 0                  507 non-null    int64  
 1   Place_Grew_Up               507 non-null    str    
 2   Price_Importance            507 non-null    str    
 3   Brand_Importance            507 non-null    str    
 4   Peer_Importance             507 non-null    str    
 5   Utility_Importance          507 non-null    str    
 6   Track_Expenditures          507 non-null    str    
 7   Expenditure_Graph           507 non-null    str    
 8   Justify_Unexpected_Expense  355 non-null    str    
 9   Unplanned_Purchases         507 non-null    int64  
 10  Peer_Influence              507 non-null    int64  
 11  Finance_Confidence          507 non-null    int64  
 12  Monthly_Spend               507 non-null    int64  
 13  Group                       507 non-null    in

In [11]:
import numpy as np
import pandas as pd
from kmodes.kprototypes import KPrototypes
from sklearn.metrics import silhouette_score
from sklearn.model_selection import KFold

def evaluate_kprototypes_custom_logic(df, target_col='Risk_score', n_folds=5):
    X_df = df.drop(columns=["Timestamp", "Group"], errors='ignore').copy()
    
    for col in X_df.columns:
        if not pd.api.types.is_numeric_dtype(X_df[col]):
            X_df[col] = X_df[col].astype(object).fillna("Unknown")
        else:
            X_df[col] = X_df[col].fillna(0).astype(float)

    spend_values = df[target_col].values
    cat_indices = [i for i, col in enumerate(X_df.columns) if X_df[col].dtype == 'object']

    kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)
    overall_results = []

    print(f"{'K':<5} | {'Avg Silhouette':<15} | {'Custom Deviation':<20}")
    print("-" * 50)

    for k in range(3, 20):
        fold_silhouettes = []
        fold_custom_deviations = []

        for train_idx, val_idx in kf.split(X_df):
            X_fold = X_df.iloc[train_idx]
            spend_fold = spend_values[train_idx]

            kp = KPrototypes(n_clusters=k, init='Cao', n_init=1, random_state=42)
            labels = kp.fit_predict(X_fold.values, categorical=cat_indices)

            # Silhouette on numeric cols only
            try:
                sil = silhouette_score(X_fold.select_dtypes(include=[np.number]), labels)
            except:
                sil = 0
            fold_silhouettes.append(sil)

            # (sum(|M - means|^0.5))^2
            cluster_means = [spend_fold[labels == cid].mean() for cid in np.unique(labels) if np.any(labels == cid)]
            c_means = np.array(cluster_means)
            M = np.mean(c_means)
            diffs = np.abs(M - c_means)
            custom_dev = (np.sum(diffs**0.5))**2
            fold_custom_deviations.append(custom_dev)

        avg_sil = np.mean(fold_silhouettes)
        avg_dev = np.mean(fold_custom_deviations)
        overall_results.append({'k': k, 'Silhouette': avg_sil, 'Custom_Deviation': avg_dev})
        print(f"{k:<5} | {avg_sil:<15.4f} | {avg_dev:<20.2f}")

    return pd.DataFrame(overall_results)

kproto_custom_results = evaluate_kprototypes_custom_logic(data)

K     | Avg Silhouette  | Custom Deviation    
--------------------------------------------------
3     | 0.4798          | 10.36               
4     | 0.4185          | 42.51               
5     | 0.3763          | 211.94              
6     | 0.3704          | 675.95              
7     | 0.3861          | 1057.23             
8     | 0.3747          | 1734.80             
9     | 0.3779          | 2372.59             
10    | 0.3972          | 3261.15             
11    | 0.3892          | 3787.13             
12    | 0.3951          | 4693.91             
13    | 0.3990          | 5452.04             
14    | 0.3929          | 6610.04             
15    | 0.4229          | 6695.60             
16    | 0.4284          | 7997.64             
17    | 0.4487          | 8810.46             
18    | 0.4492          | 10174.71            
19    | 0.4665          | 9532.59             


In [12]:
import pandas as pd
import numpy as np
from kmodes.kprototypes import KPrototypes
from kmodes.kmodes import KModes
from sklearn.metrics import normalized_mutual_info_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold

# ── 1. Prepare Data ───────────────────────────────────────────────────────────
drop_list = ['Timestamp', 'Group',"Unnamed: 0"]
X_full = data.drop(columns=drop_list, errors='ignore').copy()
y_true = data['Group'].values

# Fix dtypes for K-Prototypes
for col in X_full.columns:
    if not pd.api.types.is_numeric_dtype(X_full[col]):
        X_full[col] = X_full[col].astype(object).fillna("Missing")
    else:
        X_full[col] = X_full[col].astype(float).fillna(0)

# ── 2. CV Function ────────────────────────────────────────────────────────────
def get_clustering_nmi_cv(df_input, n_clusters=7, n_folds=5):
    """Calculates Mean NMI using Stratified K-Fold for stability.
    Auto-switches to KModes if all columns are categorical."""
    
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)
    fold_nmis = []

    cat_indices = [i for i, col in enumerate(df_input.columns)
                   if df_input[col].dtype == 'object']
    
    all_categorical = len(cat_indices) == len(df_input.columns)  # switch flag

    for _, val_idx in skf.split(df_input, y_true):
        X_fold = df_input.iloc[val_idx].copy()
        y_fold = y_true[val_idx]

        # Scale numeric columns
        num_cols = X_fold.select_dtypes(include=[np.number]).columns
        if not num_cols.empty:
            scaler = StandardScaler()
            X_fold[num_cols] = scaler.fit_transform(X_fold[num_cols])

        # Auto-switch to KModes if no numeric columns remain
        if all_categorical:
            kp = KModes(n_clusters=n_clusters, init='Cao', n_init=2, random_state=42)
            clusters = kp.fit_predict(X_fold.values)
        else:
            kp = KPrototypes(n_clusters=n_clusters, init='Cao', n_init=2, random_state=42, n_jobs=-1)
            clusters = kp.fit_predict(X_fold.values, categorical=cat_indices)

        fold_nmis.append(normalized_mutual_info_score(y_fold, clusters))

    return np.mean(fold_nmis)

# ── 3. Baseline ───────────────────────────────────────────────────────────────
print("Calculating Baseline NMI with 5-Fold Cross-Validation...")
baseline_nmi = get_clustering_nmi_cv(X_full)
print(f"📊 BASELINE CV-NMI (All Features): {baseline_nmi:.4f}")
print("-" * 65)

# ── 4. Drop-One-Out Loop ──────────────────────────────────────────────────────
results = []
for col in X_full.columns:
    X_temp = X_full.drop(columns=[col])
    current_nmi = get_clustering_nmi_cv(X_temp)

    improvement = current_nmi - baseline_nmi
    status = "⭐ NOISE (Drop)" if improvement > 0 else "✅ KEEP"

    results.append({
        "Feature_Removed": col,
        "CV_NMI_Without":  current_nmi,
        "Change":          improvement,
        "Status":          status
    })

    print(f"Removed: {col:<25} | CV-NMI: {current_nmi:.4f} | Change: {improvement:+.4f} | {status}")

# ── 5. Summary ────────────────────────────────────────────────────────────────
results_df = pd.DataFrame(results).sort_values(by="CV_NMI_Without", ascending=False)

print("\n--- Feature Impact Summary (Ranked by NMI Improvement) ---")
print(results_df.to_string(index=False))

best_to_drop = results_df[results_df['Change'] > 0]['Feature_Removed'].tolist()
print(f"\n💡 Consider dropping these noisy features: {best_to_drop}")

Calculating Baseline NMI with 5-Fold Cross-Validation...
📊 BASELINE CV-NMI (All Features): 0.5688
-----------------------------------------------------------------
Removed: Place_Grew_Up             | CV-NMI: 0.5776 | Change: +0.0088 | ⭐ NOISE (Drop)
Removed: Price_Importance          | CV-NMI: 0.5705 | Change: +0.0017 | ⭐ NOISE (Drop)
Removed: Brand_Importance          | CV-NMI: 0.5653 | Change: -0.0034 | ✅ KEEP
Removed: Peer_Importance           | CV-NMI: 0.5641 | Change: -0.0046 | ✅ KEEP
Removed: Utility_Importance        | CV-NMI: 0.5809 | Change: +0.0121 | ⭐ NOISE (Drop)
Removed: Track_Expenditures        | CV-NMI: 0.6027 | Change: +0.0339 | ⭐ NOISE (Drop)
Removed: Expenditure_Graph         | CV-NMI: 0.6095 | Change: +0.0407 | ⭐ NOISE (Drop)
Removed: Justify_Unexpected_Expense | CV-NMI: 0.5650 | Change: -0.0038 | ✅ KEEP
Removed: Unplanned_Purchases       | CV-NMI: 0.5270 | Change: -0.0418 | ✅ KEEP
Removed: Peer_Influence            | CV-NMI: 0.5823 | Change: +0.0135 | ⭐ NOISE (Dro

In [13]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from kmodes.kprototypes import KPrototypes

# 1. FINAL MODEL FITTING (using your confirmed k=10)
# Ensure only numeric features are used for calculations while identifying categorical ones
X_df = X_full
# Sync data types for K-Prototypes
for col in X_df.columns:
    if not pd.api.types.is_numeric_dtype(X_df[col]):
        X_df[col] = X_df[col].astype(object).fillna("Unknown")
    else:
        X_df[col] = X_df[col].fillna(0).astype(float)

cat_indices = [i for i, col in enumerate(X_df.columns) if X_df[col].dtype == 'object']

# Fit the final model
kp = KPrototypes(n_clusters=10, init='Cao', n_init=5, random_state=42)
data['Cluster'] = kp.fit_predict(X_df.values, categorical=cat_indices)

# 2. GENERATE IDENTITY HEATMAPS (Behavioral DNA)
# This loops through all survey columns to create the "Confusion Matrix" style distribution
cols_to_visualize = [col for col in data.columns if col not in ['Cluster', 'Timestamp']]

for col in cols_to_visualize:
    # Create a cross-tabulation: Rows are Clusters, Columns are Survey Responses
    # 'normalize=index' makes each row sum to 1 (showing probability distribution)
    dna_matrix = pd.crosstab(data['Cluster'], data[col], normalize='index')
    
    plt.figure(figsize=(14, 8))
    sns.heatmap(dna_matrix, annot=True, cmap="YlGnBu", fmt=".2f", cbar_kws={'label': 'Concentration (%)'})
    
    plt.title(f"Identity Matrix: {col} distribution across Clusters", fontsize=16, pad=20)
    plt.ylabel("Cluster ID", fontsize=12)
    plt.xlabel(f"Student Responses for {col}", fontsize=12)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    
    # Save the files for your report
    # plt.savefig(f'Persona_DNA_{col}.png') 
    plt.show()



ValueError: Clustering algorithm could not initialize. Consider assigning the initial clusters manually.